In [1]:
#pd.set_option('display.max_columns', None)
import pandas as pd
import os 
pd.set_option('display.max_rows', None)        # show all rows
pd.set_option('display.max_columns', None)     # show all columns
pd.set_option('display.width', None)           # auto-detect width
pd.set_option('display.max_colwidth', None) 

path_name = '/Users/yerik/_apple_lib/_a_progs/_a2ms_env/_9_ML_project/data/processed/df_03_27_2026_aiff_tracks_data.pkl'

df= pd.read_pickle(path_name)

df_raw= pd.read_pickle(path_name)

print(len(df))

# check if paths exist for analized songs 

print(df['Path'].apply(lambda x: os.path.exists(x)).all())


2402
True


In [2]:
# -----######-----######-----######-----######-----######
# FINAL JUPYTER AUDIO PLAYER (STABLE + CLEAN INPUT + THREAD)
# -----######-----######-----######-----######-----######

import pygame
import random
import threading
import time
import sys
from tqdm import tqdm


# ----- CLEAN INPUT FIX -----
def _get_clean_input():

    # flush stdin (fix Jupyter ghost enter)
    try:
        import termios
        termios.tcflush(sys.stdin, termios.TCIFLUSH)
    except:
        pass

    while True:
        cmd = input("\n👉 Enter command [s/n/q]: ").strip().lower()
        if cmd != "":
            return cmd


def _audio_1104_i5_GET_df_player(df):

    # ---------------- INIT ----------------
    pygame.mixer.init()

    paths = df['Path'].dropna().tolist()

    if len(paths) == 0:
        print("❌ No valid paths found")
        return

    print(f"\n🎧 Loaded {len(paths)} tracks")

    # ----- STATE -----
    current_track = {"path": None}
    lock = threading.Lock()

    # ----- PLAYER THREAD -----
# ----- PLAYER THREAD (FIXED) -----
    def play_loop():
    
        current_playing = None
    
        while True:
    
            if current_track["path"] is not None:
    
                try:
                    current_playing = current_track["path"]
    
                    pygame.mixer.music.load(current_playing)
                    pygame.mixer.music.play()
    
                    # ---- LOOP BUT ALLOW INTERRUPT ----
                    while True:
    
                        # if new track requested → BREAK IMMEDIATELY
                        if current_track["path"] != current_playing:
                            pygame.mixer.music.stop()
                            break
    
                        if not pygame.mixer.music.get_busy():
                            break
    
                        time.sleep(0.1)
    
                except Exception as e:
                    print(f"❌ Error: {e}")
    
            time.sleep(0.05)
    # start thread
    threading.Thread(target=play_loop, daemon=True).start()

    # ----- INIT BAR -----
    for _ in tqdm(range(50), desc="Initializing Player"):
        time.sleep(0.01)

    print("\n🎛 Controls:")
    print("   s → start")
    print("   n → next")
    print("   q → quit")

    # ---------------- MAIN LOOP ----------------
    while True:

        cmd = _get_clean_input()

        # ----- START -----
        if cmd in ["s", "start"]:
            path = random.choice(paths)

            with lock:
                pygame.mixer.music.stop()
                current_track["path"] = path

            print(f"\n▶️ NOW PLAYING:\n{path}")

        # ----- NEXT -----
        elif cmd in ["n", "next"]:
            path = random.choice(paths)

            with lock:
                pygame.mixer.music.stop()
                current_track["path"] = path

            print(f"\n⏭️ NEXT TRACK:\n{path}")

        # ----- QUIT -----
        elif cmd in ["q", "quit"]:
            pygame.mixer.music.stop()
            print("\n🛑 Player stopped")
            break

        else:
            print("⚠️ Use: s / n / q")

pygame 2.6.1 (SDL 2.28.4, Python 3.11.11)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [3]:
# !#!#!#!#! RUNNING STATEMENTS !#!#!#!#!

_audio_1104_i5_GET_df_player(df)


🎧 Loaded 2402 tracks


Initializing Player: 100%|████████████████████████████████████████████████████| 50/50 [00:00<00:00, 80.83it/s]



🎛 Controls:
   s → start
   n → next
   q → quit



👉 Enter command [s/n/q]:  s



▶️ NOW PLAYING:
/Users/yerik/Music/_1_NEW_SOURCE/_2025_this/__25_12_Minimal_jazzy_Housy/dylu_3U[25]-123BPM-12B_Emaj--id_t1-12190e---Deep-ZRECORDS--by--MISTURAKENDRAC-smilefeatkendr(U)-2014.aiff


...


👉 Enter command [s/n/q]:  n



⏭️ NEXT TRACK:
/Users/yerik/Music/_1_NEW_SOURCE/_2023_this/_23_08_SL_ACAPELLAS/dylu_0S[23]-126BPM-9A_Emin--id_t9-580e---Tech-MINUS--by--DJMINX-awalkinthepa(O)-2004.aiff



👉 Enter command [s/n/q]:  n



⏭️ NEXT TRACK:
/Users/yerik/Music/_1_NEW_SOURCE/_2024_this/__24_06_Metroplex_1_and_RNDMgood/dylu_0V[24]-126BPM-12B_Emaj--id_t28-581a---Tech-ETRURIAB--by--BASTINOV-regisoriginalm(O)-2016.aiff


KeyboardInterrupt: Interrupted by user

In [2]:
# -----######-----######-----######-----######-----######
# AUDIO PLAYER (THREAD SAFE + WORKS IN JUPYTER)
# -----######-----######-----######-----######-----######

import pygame
import random
import threading
import time
from tqdm import tqdm

def _audio_1104_i4_GET_df_player(df):

    pygame.mixer.init()

    paths = df['Path'].dropna().tolist()

    if len(paths) == 0:
        print("❌ No valid paths found")
        return

    print(f"\n🎧 Loaded {len(paths)} tracks\n")

    current_track = {"path": None}
    lock = threading.Lock()

    # ----- PLAYER THREAD -----
    def play_loop():
        while True:
            if current_track["path"] is not None:
                try:
                    pygame.mixer.music.load(current_track["path"])
                    pygame.mixer.music.play()

                    while pygame.mixer.music.get_busy():
                        time.sleep(0.2)

                except Exception as e:
                    print(f"❌ Error: {e}")

                current_track["path"] = None
            else:
                time.sleep(0.2)

    # start background player
    t = threading.Thread(target=play_loop, daemon=True)
    t.start()

    # fake tqdm init
    for _ in tqdm(range(50), desc="Initializing Player"):
        time.sleep(0.01)

    print("\nControls → [s] start | [n] next | [q] quit")

    # ----- MAIN LOOP -----
    while True:

        cmd = input("\n👉 Enter command: ").lower()

        if cmd == "s":
            path = random.choice(paths)

            with lock:
                pygame.mixer.music.stop()
                current_track["path"] = path

            print(f"\n▶️ NOW PLAYING:\n{path}")

        elif cmd == "n":
            path = random.choice(paths)

            with lock:
                pygame.mixer.music.stop()
                current_track["path"] = path

            print(f"\n⏭️ NEXT:\n{path}")

        elif cmd == "q":
            pygame.mixer.music.stop()
            print("\n🛑 Player stopped")
            break

        else:
            print("⚠️ Use only: s / n / q")

pygame 2.6.1 (SDL 2.28.4, Python 3.11.11)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [ ]:
# !#!#!#!#! RUNNING STATEMENTS !#!#!#!#!

_audio_1104_i4_GET_df_player(df)


🎧 Loaded 2402 tracks



Initializing Player: 100%|████████████████████████████████████████████████████| 50/50 [00:00<00:00, 81.37it/s]



Controls → [s] start | [n] next | [q] quit



👉 Enter command:  s



▶️ NOW PLAYING:
/Users/yerik/Music/_1_NEW_SOURCE/_2025_this/__25_07_SpotLite_BP_90s_Latin_BFBFBF/dylu_0W[25]-129BPM-10B_Dmaj--id_t1-7220j---Hous-PIAS]ÉLE--by--CECEPENISTONLA-finallyoriginal(O)-2025.aiff


...


⏭️ NEXT:
/Users/yerik/Music/_1_NEW_SOURCE/_2025_this/__25_06_ZSOOJ_BDAY_cancer/dylu_0T[25]-126BPM-10B_Dmaj--id_t1-6280q---Mini-SALOMONR--by--CHKLTEBLACKTUE-facilitypetarc(R)-2019.aiff



👉 Enter command:  n



⏭️ NEXT:
/Users/yerik/Music/_1_NEW_SOURCE/_2025_this/__25_12_Minimal_jazzy_Housy/dylu_0U[25]-120BPM-8A_Amin--id_t1-12190d---Deep-VIBEBOUT--by--VICKLAVENDERPE-thelovesong(U)-2018.aiff



👉 Enter command:  n



⏭️ NEXT:
/Users/yerik/Music/_1_NEW_SOURCE/_2025_this/__25_05_VMNT_puma_DeepHouse/dylu_0S[25]-123BPM-6A_Gmin--id_t1-52308---Deep-DIAPHAN--by--VSEXION-kadonoriginalm(O)-2010.aiff


In [10]:
df['Path'].head()

0       /Users/yerik/Music/_1_NEW_SOURCE/_2026_this/__26_01_Mini_VOCAL/dylu_0T[26]-123BPM-9A_Emin--id_t1-12306---Deep-PHONOGRAM--by--RICKWADE-functionalanger(O)-2025.aiff
1    /Users/yerik/Music/_1_NEW_SOURCE/_2026_this/__26_01_Mini_VOCAL/dylu_0T[26]-123BPM-6A_Gmin--id_t1-12300---Deep-CDR(CROS--by--AARONCARLDYED-nakedfeataaron(O)-2009.aiff
2    /Users/yerik/Music/_1_NEW_SOURCE/_2026_this/__26_01_Mini_VOCAL/dylu_6T[26]-126BPM-7B_Fmaj--id_t1-12304---Deep-CDR(CROS--by--DYEDSOUNDOROM-beautifulevaor(O)-2009.aiff
3             /Users/yerik/Music/_1_NEW_SOURCE/_2026_this/__26_01_Mini_VOCAL/dylu_1U[26]-123BPM-8B_Cmaj--id_t1-12305---Tech-GETPHYSI--by--PEZZNER-iamyoudjtre(R)-2025.aiff
4     /Users/yerik/Music/_1_NEW_SOURCE/_2026_this/__26_01_Mini_VOCAL/dylu_0V[26]-126BPM-11A_F#min--id_t1-12302---Tech-CROSSTOWN--by--PIERBUCCI-hayconsuelosam(R)-2006.aiff
Name: Path, dtype: object

In [ ]:
from playsound import playsound
playsound("/Users/yerik/Music/_1_NEW_SOURCE/_2026_this/__26_01_Mini_VOCAL/dylu_0T[26]-123BPM-9A_Emin--id_t1-12306---Deep-PHONOGRAM--by--RICKWADE-functionalanger(O)-2025.aiff")